# Operations Decision Model — Colab quickstart

Runs the full verification suite (65 checks across 4 layers) and the
5-million-row pipeline scale test in the browser, no setup.

**Option A (GitHub, once the private repo exists):** fill in the repo
URL below. **Option B (zip):** upload `opsproject_phase2_build2.zip`
via the folder icon on the left, then run the unzip cell.

In [ ]:
# Option A — clone the private repo (paste a fine-grained token when asked)
# !git clone https://github.com/<your-username>/operations-decision-model.git opsproject

# Option B — you uploaded the zip:
!unzip -oq opsproject_phase2_build2.zip
%cd opsproject
!pip install -q -r requirements.txt

In [ ]:
# The full verification suite: writes docs/VERIFICATION_REPORT.md
!python tests/run_verification.py

In [ ]:
# The 5M-row pipeline scale test
!python pipeline/scale_test.py

In [ ]:
# Read the generated report inline
from IPython.display import Markdown
Markdown(open('docs/VERIFICATION_REPORT.md').read())

## Use the model directly
The cell below prices a 3-station operation and values two changes.

In [ ]:
import sys; sys.path.insert(0, 'model')
import opsmodel as M
from dataclasses import replace

op = M.Operation(
    stations=(M.Station('pick', 8.0, 3, service_cv=0.8),
              M.Station('pack', 13.0, 2, service_cv=0.6),
              M.Station('load', 25.0, 1)),
    external_arrivals=(20.0, 0.0, 0.0),
    routing=((0,1,0),(0,0,1),(0,0,0)),
    margin=9.0, waiting_cost=4.0, hourly_overhead=120.0)
b = M.evaluate(op)
print(f'Profit: ${b.profit_per_hour:,.2f}/hr   bottleneck: {b.bottleneck}')

def faster(i, f):
    def ap(o):
        s = list(o.stations); s[i] = replace(s[i], service_rate=s[i].service_rate*f)
        return o.with_changes(stations=tuple(s))
    return ap
a = M.Change('re-slot picking', faster(0, 1.25), install_cost=3000)
c = M.Change('faster packing', faster(1, 1.30), install_cost=5000)
best, ranked = M.best_plan(op, [a, c], budget=100_000)
for cv in ranked:
    print(f"{' + '.join(cv.names) or 'do nothing':<28} V = ${cv.value_per_year:>10,.0f}/yr")